# Проверка «фикс не whitespace-only» (уровень 1, пункт 4)

**Идея:** если в фикс-коммите функция не изменилась по существу (идентична, менялись только пробелы или только комментарии) — она попала в BigVul случайно (ловушка из DiverseVul).

**Нужно:** исходный BigVul с колонкой `func_after` (та самая, что была снесена при чистке). Если исходника нет — скачать тот же релиз заново и НЕ удалять поля.

**Выход:** `data/processed/dedup_input_clean.csv` — без подозрительных строк.

## 1. Загрузка и джойн

Ключ джойна: `cve + commit_id + func_before` (один коммит может трогать несколько функций).

In [ ]:
import pandas as pd
from datasets import load_dataset

DEDUP_IN = "data/processed/dedup_input.csv"

# 1. Наш файл
cur = pd.read_csv(DEDUP_IN)
print("cur:", cur.shape, cur.columns.tolist())
print(cur.head(3))

# 2. BigVul
ds = load_dataset("bstee615/bigvul")

if hasattr(ds, "keys"):
    print("splits:", list(ds.keys()))
    split = "train" if "train" in ds else list(ds.keys())[0]
    ds = ds[split]

# 3. В DataFrame
orig = ds.to_pandas()
print("orig:", orig.shape, orig.columns.tolist())
print(orig.head(3))

# 4. Приводим имена колонок к нижнему регистру + алиасы
orig = orig.rename(columns={
    "CVE ID": "cve",
    "cve_id": "cve",
    "commit": "commit_id",
})
orig.columns = [c.lower() for c in orig.columns]
print("orig cols after:", orig.columns.tolist())

need = ["cve", "commit_id", "func_before", "func_after"]
missing = [c for c in need if c not in orig.columns]
assert not missing, f"в исходнике нет колонок: {missing}"

# 5. Нормализуем ключ cve (BigVul хранит CVE-2019-1234 в верхнем регистре)
def norm_cve(s):
    return (
        s.astype(str)
         .str.strip()
         .str.upper()
         .str.replace("_", "-", regex=False)
    )

cur["cve"]  = norm_cve(cur["cve"])
orig["cve"] = norm_cve(orig["cve"])

# 6. Смотрим, что с кардинальностью ключа в orig
key = ["cve", "commit_id", "func_before"]
dups = orig.duplicated(subset=key).sum()
print("дубликатов ключа в orig:", dups)
print("уникальных ключей в orig:", orig[key].drop_duplicates().shape[0], "из", len(orig))

# 7. Мержим. Если в orig всё ещё есть дубли по ключу — сначала дедуп.
orig_u = orig.drop_duplicates(subset=key)[key + ["func_after"]]
print("orig_u:", orig_u.shape)

m = cur.merge(
    orig_u,
    on=key,
    how="left",
    validate="m:1",
    indicator=True,
)
print(f"строк: {len(m)}")
print(f"не нашлось в исходнике: {(m['_merge'] == 'left_only').sum()}")

m = m.drop(columns=["_merge"])
print(m.head(2))

cur: (5384, 11) ['cve', 'cwe', 'project', 'commit_id', 'lang', 'func_before', 'split', 'cwe_id', 'in_bfcwe', 'norm', 'hash']
              cve      cwe  project                                 commit_id  \
0  CVE-2017-11462  CWE-415     krb5  56f7b1bc95a2a3eeb420e069e7655fb181ade5cf   
1   CVE-2018-6063  CWE-787   Chrome  673ce95d481ea9368c4d4d43ac756ba1d6d9e608   
2  CVE-2017-13031  CWE-125  tcpdump  2d669862df7cd17f539129049f6fb70d17174125   

  lang                                        func_before  split  cwe_id  \
0    C  gss_get_mic (minor_status,\n\t     context_han...  train   415.0   
1    C  void VideoCaptureImpl::OnBufferCreated(int32_t...  train   787.0   
2    C  frag6_print(netdissect_options *ndo, register ...  train   125.0   

   in_bfcwe                                               norm  \
0      True  gss_get_mic (minor_status, context_handle, qop...   
1      True  void VideoCaptureImpl::OnBufferCreated(int32_t...   
2      True  frag6_print(netdissect_options *nd

## 2. Классификация

In [ ]:
def strip_ws(code):
    return re.sub(r"\s+", "", str(code))

def strip_ws_comments(code):
    code = re.sub(r"/\*.*?\*/", "", str(code), flags=re.S)
    code = re.sub(r"//[^\n]*", "", code)
    return re.sub(r"\s+", "", code)

def classify(before, after):
    if pd.isna(after):
        return "deleted-in-fix"   # функция удалена в фиксе -> затронута, оставляем
    if before == after:
        return "untouched"        # вообще без изменений
    if strip_ws(before) == strip_ws(after):
        return "ws-only"          # менялись только пробелы/форматирование
    if strip_ws_comments(before) == strip_ws_comments(after):
        return "comment-only"     # менялись только комментарии
    return "real-fix"

m["fix_status"] = [classify(b, a) for b, a in zip(m["func_before"], m["func_after"])]
print(m["fix_status"].value_counts().to_string())

## 3. Отчёт и чистый выход

In [ ]:
suspicious = ["untouched", "ws-only", "comment-only"]
n_susp = int(m["fix_status"].isin(suspicious).sum())
print(f"подозрительных строк: {n_susp} ({n_susp / len(m):.1%})")

# глазами: до 5 случайных подозрительных
samp = m[m["fix_status"].isin(suspicious)]
samp = samp.sample(min(5, len(samp)), random_state=0)
for _, r in samp.iterrows():
    print("="*60, f"\n{r['fix_status']} | {r['cwe']} | {r['project']} | {r['cve']}")

clean = m[~m["fix_status"].isin(suspicious)].drop(columns=["func_after", "fix_status"])
clean.to_csv("data/processed/dedup_input_clean.csv", index=False)
print(f"\ndedup_input_clean.csv: {len(clean)} строк (было {len(m)})")